In [0]:
# ============================================================
# 05_MANAGER_CANDIDATE_SCORING — REVISED GLOBAL SCORING
# Firmable | Rule-based only | Ground truth is NOT used
#
# Input : workspace.firmable_v2_silver.manager_candidates
# Output: workspace.firmable_v2_silver.manager_candidate_scores
#
# Compatibility contract:
# - Preserves all input candidate columns.
# - Preserves the columns required by Notebook 06:
#   candidate_manager_id, candidate_score, candidate_rank,
#   score_margin, ambiguity_flag, manager_seniority,
#   relationship_type.
# - Adds explanatory columns without removing existing columns.
# - candidate_score is a rule-based score, NOT a probability.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER = "workspace.firmable_v2_silver"
INPUT_TABLE = f"{SILVER}.manager_candidates"
OUTPUT_TABLE = f"{SILVER}.manager_candidate_scores"
AMBIGUITY_MARGIN = 8

# ============================================================
# 1. LOAD AND CHECK INPUT SCHEMA
# ============================================================

candidates = spark.table(INPUT_TABLE)

required_columns = {
    "company_id", "employee_id", "candidate_manager_id",
    "employee_seniority", "manager_seniority",
    "relationship_type", "candidate_priority",
    "employee_team_id", "manager_team_id",
    "employee_department", "manager_department",
    "employee_business_unit", "manager_business_unit",
    "manager_authority", "org_structure"
}
missing_columns = required_columns - set(candidates.columns)
if missing_columns:
    raise ValueError(
        f"Notebook 05 stopped. Missing candidate columns: {sorted(missing_columns)}"
    )

candidate_count = candidates.count()
print("Candidate records loaded:", candidate_count)

if candidate_count == 0:
    raise ValueError("No manager candidates found. Run and validate Notebook 04 first.")

# ============================================================
# 2. SENIORITY GAP
# A positive gap means the proposed manager is more senior.
# ============================================================

scored = candidates.withColumn(
    "seniority_gap",
    F.col("employee_seniority") - F.col("manager_seniority")
)

# ============================================================
# 3. CLASSIFY DIRECT VS FALLBACK RELATIONSHIPS
# This is explanatory metadata; it does not use ground truth.
# ============================================================

fallback_relationships = [
    "Team Manager to Individual Contributor",
    "Department Head to Individual Contributor",
    "Department Head to Team Lead",
    "CEO to Department Head - Fallback"
]

scored = (
    scored
    .withColumn(
        "relationship_tier",
        F.when(F.col("relationship_type").isin(*fallback_relationships), "FALLBACK")
         .otherwise("DIRECT_OR_STRUCTURE_DEFINED")
    )
    .withColumn(
        "relationship_scope",
        F.when(
            F.col("employee_team_id").isNotNull() &
            F.col("manager_team_id").isNotNull() &
            (F.col("employee_team_id") == F.col("manager_team_id")),
            "SAME_TEAM"
        )
        .when(
            F.col("employee_department").isNotNull() &
            F.col("manager_department").isNotNull() &
            (F.col("employee_department") == F.col("manager_department")),
            "SAME_DEPARTMENT"
        )
        .when(
            F.col("employee_business_unit").isNotNull() &
            F.col("manager_business_unit").isNotNull() &
            (F.col("employee_business_unit") == F.col("manager_business_unit")),
            "SAME_BUSINESS_UNIT"
        )
        .otherwise("COMPANY_LEVEL_OR_FUNCTIONAL")
    )
)

# ============================================================
# 4. STRUCTURAL RELATIONSHIP FIT
# Higher scores represent stronger structural reporting evidence.
#
# Direct / structure-defined relationships retain the established
# weights. Fallback weights remain lower.
# Department Head -> IC is raised from 10 to 30 because Notebook 04
# now emits this candidate only when no closer same-team Team Lead
# or Team Manager exists, and only with same-department evidence.
# At the existing Notebook 06 threshold of 55, it still requires
# corroborating alignment, authority and seniority evidence.
# ============================================================

scored = scored.withColumn(
    "relationship_fit_score",
    F.when(
        F.col("relationship_type") == "Functional Executive to Department Head", 49
    )
    .when(
        F.col("relationship_type").isin(
            "Team Lead to Individual Contributor",
            "Team Manager to Team Lead",
            "Department Head to Team Manager",
            "Division President to Department Head",
            "CEO to C-Suite Executive",
            "CEO to Division President",
            "CEO to Department Head - Flat"
        ), 45
    )
    .when(
        (F.col("relationship_type") == "Department Head to Team Lead") &
        (F.col("org_structure") == "Flat"), 45
    )
    .when(
        F.col("relationship_type") == "Team Manager to Individual Contributor", 24
    )
    .when(
        F.col("relationship_type") == "Department Head to Team Lead", 22
    )
    .when(
        F.col("relationship_type") == "CEO to Department Head - Fallback", 18
    )
    .when(
        F.col("relationship_type") == "Department Head to Individual Contributor", 30
    )
    .otherwise(F.lit(None).cast("int"))
)

# Fail closed if Notebook 04 emits an unrecognized relationship type.
unknown_relationships = (
    scored.filter(F.col("relationship_fit_score").isNull())
    .select("relationship_type").distinct().collect()
)
if unknown_relationships:
    unknown_values = [r["relationship_type"] for r in unknown_relationships]
    raise ValueError(
        f"Unrecognized relationship types from Notebook 04: {unknown_values}"
    )

# ============================================================
# 5. TEAM ALIGNMENT SCORE
# ============================================================

scored = scored.withColumn(
    "team_alignment_score",
    F.when(
        F.col("employee_team_id").isNotNull() &
        F.col("manager_team_id").isNotNull() &
        (F.col("employee_team_id") == F.col("manager_team_id")),
        15
    ).otherwise(0)
)

# ============================================================
# 6. DEPARTMENT ALIGNMENT SCORE
# ============================================================

scored = scored.withColumn(
    "department_alignment_score",
    F.when(
        F.col("employee_department").isNotNull() &
        F.col("manager_department").isNotNull() &
        (F.col("employee_department") == F.col("manager_department")),
        12
    ).otherwise(0)
)

# ============================================================
# 7. BUSINESS UNIT ALIGNMENT SCORE
# ============================================================

scored = scored.withColumn(
    "business_unit_alignment_score",
    F.when(
        F.col("employee_business_unit").isNotNull() &
        F.col("manager_business_unit").isNotNull() &
        (F.col("employee_business_unit") == F.col("manager_business_unit")),
        8
    ).otherwise(0)
)

# ============================================================
# ============================================================
# 8. MANAGEMENT AUTHORITY SUPPORTING EVIDENCE
# CEO receives stronger authority evidence.
# Other executive and manager weights remain unchanged.
# ============================================================

scored = scored.withColumn(
    "authority_score",
    F.when(
        F.col("manager_seniority") == 0,
        10
    )
    .when(
        F.col("manager_authority") == "People Manager",
        8
    )
    .when(
        F.col("manager_authority") == "Potential People Manager",
        5
    )
    .when(
        F.col("manager_authority") == "Executive",
        4
    )
    .otherwise(0)
)

# ============================================================
# 9. SENIORITY GAP SUPPORTING EVIDENCE
# Smaller positive gaps are preferred as nearer reporting levels.
# ============================================================

scored = scored.withColumn(
    "seniority_score",
    F.when(F.col("seniority_gap").between(1, 2), 5)
     .when(F.col("seniority_gap").between(3, 4), 3)
     .when(F.col("seniority_gap").between(5, 6), 2)
     .when(F.col("seniority_gap") > 6, 1)
     .otherwise(0)
)

# ============================================================
# 10. BUILD EXPLICIT SCORE EVIDENCE
# This makes every score explainable in downstream review.
# ============================================================

scored = scored.withColumn(
    "score_evidence",
    F.concat_ws(
        "; ",
        F.concat(F.lit("relationship="), F.col("relationship_type")),
        F.concat(F.lit("scope="), F.col("relationship_scope")),
        F.when(F.col("team_alignment_score") > 0, F.lit("same_team")),
        F.when(F.col("department_alignment_score") > 0, F.lit("same_department")),
        F.when(F.col("business_unit_alignment_score") > 0, F.lit("same_business_unit")),
        F.when(F.col("authority_score") > 0, F.concat(F.lit("manager_authority="), F.col("manager_authority"))),
        F.concat(F.lit("seniority_gap="), F.col("seniority_gap").cast("string"))
    )
)

# ============================================================
# 11. CALCULATE RULE-BASED CANDIDATE SCORE
# Score is capped at 100 and is NOT a probability.
# ============================================================

score_components = [
    "relationship_fit_score",
    "team_alignment_score",
    "department_alignment_score",
    "business_unit_alignment_score",
    "authority_score",
    "seniority_score"
]

scored = scored.withColumn(
    "candidate_score",
    F.least(
        F.lit(100),
        sum(F.col(column) for column in score_components)
    ).cast("int")
)

# ============================================================
# 12. RANK CANDIDATES PER EMPLOYEE
# Prefer stronger score, then Notebook 04 priority, then nearest
# seniority gap, then stable manager ID tie-break.
# ============================================================

ranking_window = (
    Window.partitionBy("company_id", "employee_id")
    .orderBy(
        F.col("candidate_score").desc(),
        F.col("candidate_priority").asc(),
        F.col("seniority_gap").asc(),
        F.col("candidate_manager_id").asc()
    )
)

scored = scored.withColumn(
    "candidate_rank", F.row_number().over(ranking_window)
)

# ============================================================
# 13. TOP / SECOND SCORE AND AMBIGUITY
# Margin < 8 remains the established ambiguity rule.
# ============================================================

score_window = (
    Window.partitionBy("company_id", "employee_id")
    .orderBy(
        F.col("candidate_score").desc(),
        F.col("candidate_priority").asc(),
        F.col("seniority_gap").asc(),
        F.col("candidate_manager_id").asc()
    )
)

scored = (
    scored
    .withColumn("top_candidate_score", F.first("candidate_score").over(score_window))
    .withColumn("second_candidate_score", F.lead("candidate_score", 1).over(score_window))
    .withColumn(
        "score_margin",
        F.when(
            F.col("second_candidate_score").isNotNull(),
            F.col("top_candidate_score") - F.col("second_candidate_score")
        ).otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "ambiguity_flag",
        F.when(
            F.col("score_margin").isNotNull() &
            (F.col("score_margin") < AMBIGUITY_MARGIN),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

# ============================================================
# 14. VALIDATE IN MEMORY BEFORE OVERWRITING OUTPUT TABLE
# ============================================================

duplicate_pairs = (
    scored.groupBy("company_id", "employee_id", "candidate_manager_id")
    .count().filter(F.col("count") > 1).count()
)
invalid_scores = scored.filter(
    F.col("candidate_score").isNull() |
    (F.col("candidate_score") < 0) |
    (F.col("candidate_score") > 100)
).count()
invalid_seniority_gaps = scored.filter(
    F.col("seniority_gap").isNull() | (F.col("seniority_gap") <= 0)
).count()
invalid_margins = scored.filter(
    F.col("score_margin").isNotNull() & (F.col("score_margin") < 0)
).count()
invalid_ids = scored.filter(
    F.col("company_id").isNull() |
    F.col("employee_id").isNull() |
    F.col("candidate_manager_id").isNull() |
    (F.col("employee_id") == F.col("candidate_manager_id"))
).count()

print("\n========== PRE-PUBLICATION SCORING VALIDATION ==========")
print("Total candidates:", scored.count())
print("Duplicate employee-manager pairs:", duplicate_pairs)
print("Invalid candidate scores:", invalid_scores)
print("Invalid seniority gaps:", invalid_seniority_gaps)
print("Invalid score margins:", invalid_margins)
print("Invalid IDs / self relationships:", invalid_ids)

if any([
    duplicate_pairs > 0,
    invalid_scores > 0,
    invalid_seniority_gaps > 0,
    invalid_margins > 0,
    invalid_ids > 0
]):
    raise ValueError(
        "Scoring validation failed. Output table was NOT overwritten. "
        "Review the validation counts above."
    )

# ============================================================
# 15. PUBLISH SCORED CANDIDATES
# ============================================================

(
    scored.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print("\nCandidate scoring table published:", OUTPUT_TABLE)

# ============================================================
# 16. POST-PUBLICATION SUMMARY
# ============================================================

final_scores = spark.table(OUTPUT_TABLE)

print("\n========== SCORING SUMMARY ==========")
final_scores.agg(
    F.count("*").alias("total_candidates"),
    F.round(F.avg("candidate_score"), 2).alias("average_score"),
    F.min("candidate_score").alias("minimum_score"),
    F.max("candidate_score").alias("maximum_score"),
    F.sum(F.col("ambiguity_flag").cast("int")).alias("ambiguous_candidate_rows")
).show(truncate=False)

print("\nRELATIONSHIP TYPE / SCORE DISTRIBUTION")
final_scores.groupBy(
    "relationship_type", "relationship_tier", "relationship_fit_score", "candidate_score"
).count().orderBy(
    "relationship_type", "candidate_score"
).show(200, truncate=False)

print("\nCANDIDATE RANK DISTRIBUTION")
final_scores.groupBy("candidate_rank").count().orderBy("candidate_rank").show(50)

print("\nASSIGNMENT THRESHOLD READINESS (Notebook 06 currently uses 55)")
final_scores.groupBy(
    "relationship_type"
).agg(
    F.count("*").alias("candidate_rows"),
    F.sum((F.col("candidate_score") >= 55).cast("int")).alias("rows_at_or_above_55"),
    F.sum((F.col("ambiguity_flag") == False).cast("int")).alias("non_ambiguous_rows")
).orderBy("relationship_type").show(truncate=False)

print("\nOUTPUT TABLE:", OUTPUT_TABLE)
print("NOTE: candidate_score is a rule-based score, NOT a probability.")
print("NOTE: Ground truth was not read or used by this notebook.")
